# From Notebook to Production Pipeline

A notebook is an excellent tool for exploring and validating a transformation. But a notebook is not a pipeline: it has no retry logic, no dependency tracking, no scheduling, and no observability. This notebook refactors the gold layer transformations into a `MedallionPipeline` class, runs it in a three-iteration loop with new data arriving between runs, and sketches what the same pipeline looks like in Apache Airflow.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import time
import shutil
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from pyspark.sql import functions as F

sys.path.insert(0, str(Path.cwd()))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("pipeline")

In [ ]:
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

all_events = [json.loads(l) for l in open("../data/input/events.jsonl")]
batch_size = len(all_events) // 3
batches = [
    all_events[:batch_size],
    all_events[batch_size:2*batch_size],
    all_events[2*batch_size:],
]

# Seed silver with batch 1
arrow_batch1 = pa.Table.from_pandas(pd.DataFrame(batches[0]))
silver_events = catalog.create_table("silver.events", schema=arrow_batch1.schema)
silver_events.append(arrow_batch1)
print(f"Silver seeded with {len(batches[0]):,} events. Remaining batches: {len(batches)-1}")

In [ ]:
from helpers import get_spark

spark = get_spark("MedallionPipeline", gold_warehouse="../data/warehouse_gold")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")
print("Spark ready.")

## What is a Data Pipeline?

A pipeline is a sequence of tasks with explicit dependencies, error handling, and scheduling. The anatomy of a medallion pipeline run:

1. Detect new data (query silver for snapshots after the last checkpoint)
2. Run quality checks on incoming silver
3. Compute incremental gold updates
4. Validate gold quality
5. Publish to gold layer
6. Update checkpoint

Two critical properties:

- **Idempotency**: re-running the pipeline with the same checkpoint must produce the same result. If a run is interrupted and retried, no data is duplicated.
- **Observability**: log each step's inputs, outputs, and timing. You must be able to answer "why did the gold row count drop at 03:15?" from logs alone, without re-running the pipeline.

## Why Notebooks Are Not Pipelines

- **No retry**: if a cell fails, you manually re-run it. In production, transient S3 errors or Spark OOMs must be retried automatically.
- **No dependency tracking**: if cell 8 depends on cell 5's output, running them out of order produces wrong results. A notebook has no enforced ordering beyond sequential execution.
- **No scheduling**: a notebook must be triggered manually or via an external tool (e.g., `jupyter nbconvert --to script`). There's no cron expression inside a notebook.
- **No observability**: print statements are not structured logs. You can't query "how long did the gold aggregation step take across all runs?"
- **Stateful globals**: `last_processed_snapshot` lives in the notebook's kernel. A kernel restart loses it.

Notebooks are for exploration. Pipelines are for production. The code is often the same — the wrapper is different.

In [ ]:
@dataclass
class PipelineState:
    """Persistent pipeline state stored to disk between runs."""
    last_processed_snapshot_id: Optional[int] = None
    run_count: int = 0
    total_gold_rows: int = 0

    def save(self, path: str):
        import json
        with open(path, "w") as f:
            json.dump(self.__dict__, f)

    @classmethod
    def load(cls, path: str) -> "PipelineState":
        import json
        try:
            with open(path) as f:
                return cls(**json.load(f))
        except FileNotFoundError:
            return cls()


class MedallionPipeline:
    """
    Medallion pipeline: detects new silver snapshots, computes gold,
    validates quality, and publishes.

    PyIceberg 0.11 does not expose incremental scan (from_snapshot_id) in
    Python. This demo therefore uses a full refresh: all silver data is loaded
    and gold is recomputed on each run. In production, replace the silver scan
    with Spark's incremental reader:

        spark.read.format("iceberg")
             .option("start-snapshot-id", last_snapshot_id)
             .load(silver_table.location())

    as shown in notebook 03_incremental_processing.
    """

    CHECKPOINT_PATH = "../data/pipeline_checkpoint.json"

    def __init__(self, spark, catalog, silver_table_name, gold_table_name):
        self.spark = spark
        self.catalog = catalog
        self.silver_table_name = silver_table_name
        self.gold_table_name = gold_table_name
        self.state = PipelineState.load(self.CHECKPOINT_PATH)

    def _get_silver_table(self):
        return self.catalog.load_table(self.silver_table_name)

    def detect_new_data(self) -> Optional[int]:
        """Return the current snapshot ID if it differs from the last checkpoint, else None."""
        table = self._get_silver_table()
        current_snapshot = table.current_snapshot()
        if current_snapshot is None:
            return None
        current_id = current_snapshot.snapshot_id
        if current_id == self.state.last_processed_snapshot_id:
            log.info("No new data since last checkpoint.")
            return None
        log.info(f"New snapshot detected: {self.state.last_processed_snapshot_id} → {current_id}")
        return current_id

    def run_silver_quality_checks(self, row_count: int) -> bool:
        if row_count == 0:
            log.warning("Silver quality check: empty table.")
        else:
            log.info(f"Silver quality check passed: {row_count:,} rows.")
        return True

    def compute_gold(self, silver_arrow, new_snapshot_id: int) -> int:
        """Compute gold from pre-loaded silver Arrow table."""
        df = self.spark.createDataFrame(silver_arrow.to_pandas())
        gold = df.withColumn("event_hour", F.date_trunc("hour", F.to_timestamp("time"))) \
                 .groupBy("event_hour", "type") \
                 .agg(F.count("*").alias("event_count"),
                      F.lit(new_snapshot_id).cast("long").alias("source_snapshot_id"))

        gold.writeTo(self.gold_table_name).createOrReplace()
        gold_count = self.spark.sql(f"SELECT COUNT(*) as n FROM {self.gold_table_name}").collect()[0]["n"]
        log.info(f"Gold table {self.gold_table_name}: {gold_count:,} rows.")
        return gold_count

    def run_gold_quality_checks(self, gold_count: int) -> bool:
        if gold_count == 0:
            log.error("Gold quality check FAILED: empty gold table.")
            return False
        neg = self.spark.sql(
            f"SELECT COUNT(*) as n FROM {self.gold_table_name} WHERE event_count < 0"
        ).collect()[0]["n"]
        if neg > 0:
            log.error(f"Gold quality check FAILED: {neg} rows with negative event_count.")
            return False
        log.info(f"Gold quality checks passed. Rows: {gold_count:,}")
        return True

    def run(self):
        """Execute one pipeline run."""
        self.state.run_count += 1
        log.info(f"--- Pipeline run #{self.state.run_count} ---")

        new_snapshot_id = self.detect_new_data()
        if new_snapshot_id is None:
            log.info("Nothing to do.")
            return

        # Load silver data once; pass it to quality check and compute_gold
        silver_table = self._get_silver_table()
        silver_arrow = silver_table.scan().to_arrow()
        self.run_silver_quality_checks(len(silver_arrow))

        gold_count = self.compute_gold(silver_arrow, new_snapshot_id)
        if not self.run_gold_quality_checks(gold_count):
            raise RuntimeError("Gold quality check failed — checkpoint NOT advanced.")

        self.state.last_processed_snapshot_id = new_snapshot_id
        self.state.total_gold_rows = gold_count
        self.state.save(self.CHECKPOINT_PATH)
        log.info(f"Run #{self.state.run_count} complete. Checkpoint: {new_snapshot_id}")

In [ ]:
pipeline = MedallionPipeline(
    spark=spark,
    catalog=catalog,
    silver_table_name="silver.events",
    gold_table_name="local.gold.hourly_events",
)

pipeline.run()
print(f"\nAfter run 1: {pipeline.state.total_gold_rows:,} gold rows")
spark.sql("SELECT * FROM local.gold.hourly_events ORDER BY event_count DESC LIMIT 5").show()

In [ ]:
# Simulate: new batch arrives in silver
silver_table = catalog.load_table("silver.events")
arrow_batch2 = pa.Table.from_pandas(pd.DataFrame(batches[1]))
silver_table.append(arrow_batch2)
print(f"Batch 2 appended to silver: {len(batches[1]):,} new events")

pipeline.run()
print(f"\nAfter run 2: {pipeline.state.total_gold_rows:,} gold rows")

In [ ]:
print("Running with no new data (should be a no-op):")
pipeline.run()
print(f"Gold rows unchanged: {pipeline.state.total_gold_rows:,}")

In [ ]:
silver_table = catalog.load_table("silver.events")
arrow_batch3 = pa.Table.from_pandas(pd.DataFrame(batches[2]))
silver_table.append(arrow_batch3)
print(f"Batch 3 appended: {len(batches[2]):,} new events")

pipeline.run()
print(f"\nAfter run 3: {pipeline.state.total_gold_rows:,} gold rows")

In [ ]:
print("Idempotency check: re-run pipeline with same checkpoint:")
rows_before = pipeline.state.total_gold_rows
pipeline.run()
rows_after = pipeline.state.total_gold_rows
print(f"Gold rows before: {rows_before:,}")
print(f"Gold rows after:  {rows_after:,}")
print(f"Idempotent: {rows_before == rows_after}")

## Orchestration Tools

| Tool | Approach | Strengths | Weaknesses |
|---|---|---|---|
| Apache Airflow | DAG-based (tasks + dependencies) | Mature, widely adopted, Python DSL, rich integrations | Heavy setup, not naturally asset-aware |
| Prefect | Pythonic flows + tasks | Easy local dev, built-in retries, cloud UI | Smaller ecosystem than Airflow |
| Dagster | Asset-centric (thinking in tables, not tasks) | Great for medallion patterns, lineage tracking built-in | Steeper learning curve |
| dbt | SQL-first transformations + lineage | Ideal for gold layer SQL transforms | Spark integration is an add-on |

For IoT data pipelines with medallion architecture, Dagster's asset-centric model maps naturally: each gold table is an asset, each pipeline run produces a new version of the asset. That said, all three are valid choices — the important thing is that you use one.

## The Airflow Equivalent (Code Sketch)

The cell below shows what the `MedallionPipeline` steps look like as an Airflow DAG. This code is illustrative only — Airflow is not installed in this environment. The key structural point: each pipeline step becomes a `PythonOperator` task, and dependencies are declared with `>>`. State passes between tasks via XCom rather than in-memory variables.

In [ ]:
# This code is illustrative — Airflow is not installed.
# It shows how the MedallionPipeline steps map to an Airflow DAG.
AIRFLOW_SKETCH = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def detect_new_data(**context):
    """Pull last checkpoint from XCom; return new snapshot ID if available."""
    last_snap = context["ti"].xcom_pull(key="last_snapshot_id") or None
    # ... check silver table ...
    context["ti"].xcom_push(key="new_snapshot_id", value=new_snapshot_id)

def compute_gold(**context):
    last_snap = context["ti"].xcom_pull(key="last_snapshot_id")
    new_snap = context["ti"].xcom_pull(key="new_snapshot_id")
    # ... run Spark job ...

def validate_gold(**context):
    # ... run quality checks ...
    pass

def advance_checkpoint(**context):
    new_snap = context["ti"].xcom_pull(key="new_snapshot_id")
    context["ti"].xcom_push(key="last_snapshot_id", value=new_snap)

with DAG("medallion_gold_pipeline", schedule_interval="*/15 * * * *", start_date=datetime(2024, 1, 1)) as dag:
    t1 = PythonOperator(task_id="detect_new_data", python_callable=detect_new_data)
    t2 = PythonOperator(task_id="compute_gold", python_callable=compute_gold)
    t3 = PythonOperator(task_id="validate_gold", python_callable=validate_gold)
    t4 = PythonOperator(task_id="advance_checkpoint", python_callable=advance_checkpoint)

    t1 >> t2 >> t3 >> t4
'''
print("Airflow DAG sketch (illustrative, not runnable):")
print(AIRFLOW_SKETCH)

## Scheduling Strategies

| Gold table | Recommended cadence | Rationale |
|---|---|---|
| Device current state | Triggered (on new silver snapshot) | Consumers need near-real-time state |
| Hourly event counts | Micro-batch (every 15 min) | Balanced freshness vs. compute cost |
| Daily group summaries | Batch (every 1 hour) | Low urgency, high compute for full rollup |
| Feature table (ML) | Batch (daily) | ML retraining is not real-time |
| Historical archive | Weekly or on-demand | No freshness requirement |

Compaction should be scheduled separately from gold refresh — run after every N writes to the silver table, not after every pipeline run. Compaction is a maintenance operation, not a transformation.

## Production Considerations

- **Idempotency**: a pipeline run must produce the same gold output if re-run with the same checkpoint. `MedallionPipeline` achieves this via Iceberg snapshot IDs.
- **Exactly-once semantics**: the Iceberg MERGE INTO commit is atomic. Either the entire merge commits (and we advance the checkpoint) or neither happens. We never advance the checkpoint before the Iceberg commit.
- **Error handling**: the checkpoint is only advanced after quality checks pass. A failed quality check leaves the checkpoint unchanged, so the next run reprocesses the same delta.
- **Schema evolution**: if the silver schema changes (new event type, renamed field), the gold schema may need updating. Use `ALTER TABLE ... ADD COLUMN` before the next pipeline run, not during.
- **Observability**: log snapshot IDs, row counts, and timing at each step. Query the gold table's `.history` to see how it evolved over pipeline runs.
- **Catalog for production**: `SqlCatalog` with SQLite is not safe for concurrent writes. Use AWS Glue, Nessie, or a REST catalog in production.

## Review Questions

1. What makes a pipeline idempotent? Why is idempotency important for retry logic?
2. What is the difference between task-centric (Airflow) and asset-centric (Dagster) orchestration? Which maps more naturally to the medallion architecture?
3. How would you handle a pipeline run that fails in the middle of the MERGE INTO step? Is the gold table in a consistent state?
4. You need to add a new gold table (device_daily_active) to the pipeline. What steps are needed to add it without breaking the existing pipeline run?
5. When should you trigger a full gold recompute (instead of incremental)? Give two scenarios.

## Challenges

- Add error handling to `MedallionPipeline.run()`: catch exceptions, log the error, and guarantee the checkpoint is NOT advanced on failure. Write a test that deliberately causes a gold quality check failure and verifies the checkpoint is unchanged.
- Extend the pipeline to process two gold tables in the same run: hourly event counts AND daily group summaries. Both should advance the same checkpoint.
- Implement scheduling: wrap `pipeline.run()` in a loop that checks for new silver snapshots every 10 seconds (using `time.sleep(10)`). Run it for 60 seconds, appending new silver data every 20 seconds, and observe the pipeline detect and process the new batches.
- Sketch the Dagster equivalent: define two Assets (`silver_events` and `gold_hourly`) with an explicit dependency between them. The `gold_hourly` asset should only materialize when `silver_events` has a new snapshot.

## Module 3 Summary: Apache Spark and the Medallion Architecture

Key takeaways from the whole module:

- **Medallion architecture**: Bronze (raw) → Silver (validated) → Gold (aggregated, business-ready). Each layer has clear contracts. The lakehouse pattern implements this with open file formats (Parquet + Iceberg) instead of proprietary databases.
- **Spark + Iceberg**: PySpark 4.0 with the Iceberg Spark runtime reads tables written by PyIceberg (and vice versa) by path. No shared catalog required in development; use Glue or REST catalog in production.
- **Correct aggregation**: event counts are additive (safe to SUM). Averages and rates are not — carry numerator and denominator separately, compute ratios at query time. DISTINCT device counts must be re-evaluated at each hierarchy level.
- **Incremental processing**: use Iceberg snapshot IDs as checkpoints, not timestamps. Snapshot-based reads are exact, reproducible, and handle clock drift and late IoT data correctly.
- **Data quality**: Write-Audit-Publish with Iceberg branches ensures production data is never touched until quality checks pass. Add `source_snapshot_id` for lineage.
- **AI readiness**: gold tables are feature tables. Use Iceberg time travel for point-in-time feature retrieval. Compact gold summaries provide effective LLM context without passing millions of raw events.
- **Pipelines**: production pipelines are idempotent, observable, and checkpointed. The `MedallionPipeline` pattern (detect → validate silver → compute gold → validate gold → publish → checkpoint) is a repeatable template.